# A/B Testing: TF-IDF vs Sentence-BERT

## Coding Camp 2026 | Tim CC26-PSU060

## Tujuan A/B Testing
A/B Testing dilakukan untuk membandingkan performa dua metode semantic matching dalam sistem PELET, yaitu metode baseline TF-IDF dan model utama Sentence-BERT. Pengujian ini bertujuan untuk mengetahui metode yang memiliki performa terbaik dalam memahami kecocokan antara resume pelamar dan job role.

# Dataset yang Digunakan

A/B Testing menggunakan dataset utama `cleaned_training_data.csv` yang telah melalui proses data wrangling, cleaning, dan preprocessing. Dataset ini berisi informasi resume, skills, kategori pekerjaan, serta job role yang digunakan sebagai data modeling dan evaluasi semantic matching.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

df = pd.read_csv("cleaned_training_data.csv")
print(df.head())

  Resume ID                                        Resume Text  \
0   R000000  Education: Bachelor's in Computer Science Expe...   
1   R000001  Education: Master's in Microbiology Experience...   
2   R000002  Education: Apprenticeship Experience: 2 years ...   
3   R000003  Education: Bachelor's in Computer Science Expe...   
4   R000004  Education: Bachelor's in Design Experience: 3 ...   

                        Education  Experience Years  \
0  Bachelor's in Computer Science                 2   
1        Master's in Microbiology                 3   
2                  Apprenticeship                 2   
3  Bachelor's in Computer Science                 0   
4            Bachelor's in Design                 3   

                                              Skills                Job Role  \
0  Market Research|Maven|Java|REST API|Spring Boo...  Java Backend Developer   
1  Agile|Data Analysis|Precision|Problem Solving|...          Microbiologist   
2  React|Customer Service|Techni

In [ ]:
# Resume / skill text
texts = df["resume_text_clean"].astype(str)

# Label target
labels = df["Job Role"].astype(str)

resume_list = []
job_list = []
true_labels = []

job_roles = labels.unique()

for i in range(len(df)):

    resume_text = texts.iloc[i]
    correct_role = labels.iloc[i]

    # POSITIVE SAMPLE
    resume_list.append(resume_text)
    job_list.append(correct_role)
    true_labels.append(1)

    # NEGATIVE SAMPLE
    for role in job_roles:

        if role != correct_role:

            resume_list.append(resume_text)
            job_list.append(role)
            true_labels.append(0)

            break

# Model A — TF-IDF + Cosine Similarity

Model A menggunakan pendekatan tradisional berbasis keyword matching dengan metode TF-IDF (Term Frequency-Inverse Document Frequency) dan Cosine Similarity. Metode ini bekerja dengan menghitung frekuensi kata pada teks resume dan job role, kemudian mengukur tingkat kemiripan antar teks berdasarkan representasi vektor.

Kelebihan metode ini adalah proses komputasi yang ringan dan cepat. Namun, metode TF-IDF memiliki keterbatasan dalam memahami makna semantik atau konteks dari suatu kalimat karena hanya berfokus pada kemunculan kata.

In [ ]:
# MODEL A : TF-IDF + COSINE SIMILARITY
print("\n MODEL A : TF-IDF ")

tfidf = TfidfVectorizer()

combined_text = resume_list + job_list

tfidf.fit(combined_text)

resume_vectors = tfidf.transform(resume_list)
job_vectors = tfidf.transform(job_list)

tfidf_scores = []

for i in range(len(resume_list)):

    sim = cosine_similarity(
        resume_vectors[i],
        job_vectors[i]
    )[0][0]

    tfidf_scores.append(sim)

# threshold baseline
tfidf_pred = [1 if s >= 0.20 else 0 for s in tfidf_scores]



 MODEL A : TF-IDF 


# Model B — Sentence-BERT

Model B menggunakan Sentence-BERT (paraphrase-multilingual-MiniLM-L12-v2) sebagai metode semantic matching utama pada sistem PELET. Model ini mampu menghasilkan sentence embedding yang dapat memahami makna dan konteks kalimat secara lebih mendalam dibandingkan metode keyword matching tradisional.

Sentence-BERT digunakan untuk mengukur tingkat kesamaan semantik antara resume pelamar dan job role sehingga sistem dapat memberikan hasil pencocokan yang lebih relevan dan akurat.

In [ ]:
# MODEL B : SENTENCE-BERT
print("\n MODEL B : Sentence-BERT ")

model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

resume_embeddings = model.encode(
    resume_list,
    convert_to_tensor=False
)

job_embeddings = model.encode(
    job_list,
    convert_to_tensor=False
)

sbert_scores = []

for i in range(len(resume_list)):

    sim = cosine_similarity(
        [resume_embeddings[i]],
        [job_embeddings[i]]
    )[0][0]

    sbert_scores.append(sim)

# threshold Sentence-BERT
sbert_pred = [1 if s >= 0.42 else 0 for s in sbert_scores]



 MODEL B : Sentence-BERT 


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Metrik Evaluasi

Evaluasi performa kedua model dilakukan menggunakan beberapa metrik klasifikasi berikut:

- Accuracy
- Precision
- Recall
- F1-Score

Metrik tersebut digunakan untuk mengukur kemampuan model dalam melakukan pencocokan resume dan job role secara akurat.

In [ ]:
# EVALUATION FUNCTION
def evaluate_model(name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 25)
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")

# HASIL A/B TESTING
evaluate_model(
    "MODEL A - TF-IDF",
    true_labels,
    tfidf_pred
)

evaluate_model(
    "MODEL B - Sentence-BERT",
    true_labels,
    sbert_pred
)

# KESIMPULAN
print("\nKESIMPULAN")

tfidf_f1 = f1_score(true_labels, tfidf_pred)
sbert_f1 = f1_score(true_labels, sbert_pred)

if sbert_f1 > tfidf_f1:
    print("Sentence-BERT memiliki performa lebih baik.")
else:
    print("TF-IDF memiliki performa lebih baik.")


MODEL A - TF-IDF
-------------------------
Accuracy  : 0.7291
Precision : 1.0000
Recall    : 0.4583
F1-Score  : 0.6285

MODEL B - Sentence-BERT
-------------------------
Accuracy  : 0.7656
Precision : 0.9979
Recall    : 0.5323
F1-Score  : 0.6943

KESIMPULAN
Sentence-BERT memiliki performa lebih baik.


# Hasil A/B Testing

| Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| TF-IDF | 72.91% | 100% | 45.83% | 62.85% |
| Sentence-BERT | 76.56% | 99.79% | 53.23% | 69.43% |

---

# Analisis Hasil

Berdasarkan hasil pengujian riil pada kode program, model Sentence-BERT menunjukkan performa yang lebih baik dibandingkan TF-IDF pada seluruh metrik evaluasi utama. Sentence-BERT memperoleh Accuracy sebesar 76.56% dan F1-Score sebesar 69.43%, lebih tinggi dibandingkan TF-IDF yang menghasilkan Accuracy sebesar 72.91% dan F1-Score sebesar 62.85%.

Peningkatan yang paling signifikan terlihat pada metrik Recall, di mana Sentence-BERT naik menjadi 53.23% dibandingkan TF-IDF yang hanya 45.83%. Hal ini menunjukkan bahwa pendekatan semantic matching berbasis Sentence-BERT jauh lebih efektif dalam memahami konteks dan hubungan semantik antara teks resume pelamar dan job role dibandingkan metode keyword matching tradisional yang hanya mengandalkan kesamaan kosakata fisik.

---

# Kesimpulan

Dari hasil A/B Testing yang telah dilakukan, dapat disimpulkan bahwa model Sentence-BERT merupakan metode terbaik untuk implementasi semantic matching pada sistem PELET. Oleh karena itu, Sentence-BERT dipilih sebagai model utama dalam proses pencocokan resume pada aplikasi PELET karena terbukti memberikan hasil penjodohan yang lebih relevan dan akurat secara kontekstual.